# ☀️ Exercise 2 — Forecasting a solar plant in Malta
**Energy Regulation, Economics and Sustainable Energy Systems in the Mediterranean and MENA Region** · Hands-on session · BALAB, Bari

## 🎯 Your mission
A (fictional) solar plant in Malta (**maximum 8.5 MW**) must send the grid operator a forecast of its hourly production. You have **18 months** of hourly data (Jan 2023 → Jun 2024) and the **weather forecast** for each hour. Your job: build a model that predicts the **power (MW)** and find out **what it looks at**.
The **last 6 months** (Jul → Dec 2024) are hidden. At the end, the instructor gives them to you: that is the final exam.

## 🧭 How we work (45 minutes, 5 steps)
You already saw all of this in Exercise 1. Now **you** do it, and it is easy:
1. **Read** the short explanation.
2. **Ask the AI** (ChatGPT / Claude, free is fine) with the **prompt we give you**, and paste the code in the empty cell. Run it.
3. Run the **CHECK** cell: it tells you if you are right. If there is an error, paste it to the AI: "fix it".
4. Stuck? Use the **rescue cells** below.

| Step | What | Time |
|---|---|---|
| 1 | Order the data, remove duplicates, fill the gaps | 10 min |
| 2 | Look at plots, remove outliers | 10 min |
| 3 | Split in time, train XGBoost, measure the error | 10 min |
| 4 | SHAP: what does the model look at? | 8 min |
| 5 | Save the model, then the final test | 7 min |

## 🆕 What is new compared with Exercise 1
- **XGBoost** instead of Random Forest. 📘 Both use many small trees. Random Forest builds them *independently* and averages (*bagging*). XGBoost builds them *one after the other*, and every new tree tries to **fix the mistakes of the previous ones** (*boosting*).
- **Night = zero power, and that is normal.** Do not delete zeros!
- **MAPE does not work** here (at night 0 ÷ 0). We use **nMAE** = the average error as a **% of the plant size** (8.5 MW).

## 🤖 ChatGPT / Claude
Paste this **once** at the start of your chat:
```
I am a complete beginner in Python and I use Google Colab. I work in energy regulation / economics.
Please explain things in simple words and give SIMPLE code: no functions, short lines, a comment on every important line.

PROJECT: a fictional solar plant in Malta (maximum output 8.5 MW). We predict the hourly power (MW) from the weather forecast.
My table is called "df" (one row per hour). Columns: timestamp, month, day_of_week, is_holiday,
solar_elevation_deg (sun height, negative = night), ghi_forecast_wm2 (forecast sun power), cloud_cover_forecast_pct,
temperature_c, wind_speed_ms, pv_power_lag_24h_mw (power of the same hour yesterday), pv_power_mw (TARGET, in MW).
The power is really 0 at night: that is NOT an error.
I use pandas, numpy, matplotlib, shap and xgboost. I will paste you a piece of my code or an error message:
explain it simply, or fix it.
```
⚠️ The AI can be wrong. Always run the **CHECK** cells.

In [ ]:
# ===== SETUP: run this cell first. Do not change it. =====
%pip install -q shap

import os, random, json, warnings
import numpy as np                      # maths
import pandas as pd                     # tables (like Excel, but in Python)
import matplotlib.pyplot as plt         # plots
import joblib                           # to save the trained model
import shap                             # explainable AI
from xgboost import XGBRegressor        # the machine learning model (NEW: Exercise 1 used Random Forest)
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
SEED = 42                               # fixed random seed: everybody gets the same numbers
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
plt.rcParams["figure.figsize"] = (10, 4)
print("Setup done.")

In [ ]:
# ===== LOAD THE TRAINING DATA (given: do not change) =====
TRAIN_URL = "https://raw.githubusercontent.com/mariocaruso96/MENA/main/exercise_2_train.csv"
pv_raw = pd.read_csv(TRAIN_URL)

print("Rows and columns:", pv_raw.shape)
pv_raw.head()

### 🆘 Rescue cells (use only if you are stuck)
- **Rescue A** replaces Steps 1 + 2. Run it, then continue with Step 3.
- **Rescue B** replaces Step 3 (it needs the table `df` from Step 2 or Rescue A). Run it, then continue with Step 4.
Try first by yourself!

In [ ]:
# RESCUE A - replaces Steps 1 and 2
# STEP 1
df = pv_raw.copy()                                          # work on a copy

df["timestamp"] = pd.to_datetime(df["timestamp"])           # text -> real date and time

df = df.sort_values("timestamp")                        # 1) oldest hour first
df = df.drop_duplicates(subset="timestamp", keep="first")   # 2) one row per hour
df = df.reset_index(drop=True)

df["hour"] = df["timestamp"].dt.hour                    # new column: hour of the day (0-23)

# 3) fill the missing temperatures with a straight line between the neighbouring hours
df["temperature_c"] = df["temperature_c"].interpolate()


# STEP 2
# ONE rule for the outliers: power above the plant maximum (8.5 MW) OR clearly negative (below -0.1 MW)
is_outlier = (df["pv_power_mw"] > 8.5) | (df["pv_power_mw"] < -0.1)
print("Outliers found:", is_outlier.sum())

df.loc[is_outlier, "pv_power_mw"] = np.nan                  # turn them into "missing"

# We never invent the TARGET: remove the rows where the power is missing
df = df.dropna(subset=["pv_power_mw", "pv_power_lag_24h_mw"])
df = df.reset_index(drop=True)
print("Rows left:", len(df))

In [ ]:
# RESCUE B - replaces Step 3
# STEP 3 - complete the blanks ___
# 3.1 Split IN TIME ORDER: learn on the past, check on the most recent months
SPLIT_DATE = "2024-04-01"
train_part = df[df["timestamp"] < SPLIT_DATE]
val_part   = df[df["timestamp"] >= SPLIT_DATE]

# 3.2 Inputs (features) = all columns except the time stamp and the target
feature_cols = [c for c in df.columns if c not in ["timestamp", "pv_power_mw"]]
print("Features:", feature_cols)

X_train = train_part[feature_cols];  y_train = train_part["pv_power_mw"]
X_val   = val_part[feature_cols];    y_val   = val_part["pv_power_mw"]

# 3.3 Train XGBoost (given)
model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5,
                     subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=1)
model.fit(X_train, y_train)

# 3.4 Predict on the validation months (a plant cannot produce less than 0 or more than 8.5 MW)
pred_val = model.predict(X_val)
pred_val = np.clip(pred_val, 0, 8.5)


# ERRORS (given: just run it)
y = y_val.values
mae  = mean_absolute_error(y, pred_val)                   # average error, in MW
rmse = np.sqrt(mean_squared_error(y, pred_val))           # like MAE, but big errors count more
nmae = mae / 8.5 * 100                                    # MAE as % of the plant size

baseline = val_part["pv_power_lag_24h_mw"].values         # baseline = "same hour yesterday"
mae_b  = mean_absolute_error(y, baseline)
rmse_b = np.sqrt(mean_squared_error(y, baseline))
nmae_b = mae_b / 8.5 * 100

print("VALIDATION            MAE (MW)   RMSE (MW)   nMAE (%)")
print(f"XGBoost model         {mae:8.3f}   {rmse:9.3f}   {nmae:8.2f}")
print(f"Baseline (yesterday)  {mae_b:8.3f}   {rmse_b:9.3f}   {nmae_b:8.2f}")

five = (val_part["timestamp"] >= "2024-05-20") & (val_part["timestamp"] < "2024-05-25")
plt.plot(val_part.loc[five, "timestamp"], y_val[five], label="Real")
plt.plot(val_part.loc[five, "timestamp"], pred_val[five.values], label="XGBoost")
plt.plot(val_part.loc[five, "timestamp"], val_part.loc[five, "pv_power_lag_24h_mw"], label="Baseline (yesterday)", alpha=0.5)
plt.legend(); plt.ylabel("MW"); plt.title("Validation: five days"); plt.show()

---
# Step 1 — Order the data, remove duplicates, fill the gaps  *(10 min)*
📘 *Data quality: garbage in, garbage out.* A time series must be **in order**, with **one row per hour**.

First look at the table (run the cell). Notice the **missing values** (temperature, power) and the **minimum / maximum** of the power: a plant of 8.5 MW cannot produce 57 MW, or −5 MW!

In [ ]:
# LOOK at the table (given: just run it)
print(pv_raw.dtypes)
print()
print("Missing values per column:")
print(pv_raw.isna().sum())
print()
pv_raw.describe().T[["min", "mean", "max"]]

**What to do**
1. Make a **copy** of `pv_raw` called `df`.
2. Convert the column `timestamp` into a real date.
3. **Sort** by `timestamp` and **drop the duplicated hours** (keep the first). Renumber the rows.
4. Create a new column `hour` (0-23) from the timestamp.
5. Fill the **missing temperatures** by interpolation (a straight line between the neighbours).

💡 **Hints:** `pd.to_datetime`, `.sort_values`, `.drop_duplicates(subset="timestamp", keep="first")`, `.reset_index(drop=True)`, `df["timestamp"].dt.hour`, `.interpolate()`.

🤖 **Prompt for the AI (copy it)**
```
I have a pandas table pv_raw. Write simple code that: (1) makes a copy called df; (2) converts the column timestamp to datetime;
(3) sorts by timestamp, drops duplicated timestamps keeping the first, and resets the index; (4) adds a column "hour" with the hour of the day;
(5) fills the missing values of the column temperature_c with interpolate. No functions.
```

In [ ]:
# Write (or paste) your code here



In [ ]:
# CHECK (given)
print("Rows:", len(df), "   (expected 13128)")
print("Time in order?", df["timestamp"].is_monotonic_increasing, "   (expected True)")
print("Missing temperatures:", df["temperature_c"].isna().sum(), "   (expected 0)")

---
# Step 2 — Plots and outliers  *(10 min)*
📘 *Always plot before you clean.*

**What to do — 1) plots (look before you clean!).** Draw 3 plots:
- a **histogram** of `pv_power_mw`;
- a **scatter plot** of `ghi_forecast_wm2` (x) against `pv_power_mw` (y), small transparent dots, x-axis limited to 0-1200;
- the **average power for each hour** of the day (bar plot).

🤖 **Prompt for the AI**
```
With matplotlib and my table df draw 3 plots in one row: (1) histogram of pv_power_mw with 100 bins; (2) scatter plot of ghi_forecast_wm2 (x) vs pv_power_mw (y)
with s=3 and alpha=0.3 and x-axis from 0 to 1200; (3) bar plot of the average pv_power_mw for each value of the column hour. Titles included. Simple code.
```
**Look at them:** 
- **Left:** most hours are 0 (the nights), but there are **huge** values (up to ~57 MW) and **negative** ones. Impossible!
- **Middle:** more sun = more power, with a **flat ceiling at 8.5 MW** (the inverters limit the plant: real, not an error). Dots far above or below are outliers.
- **Right:** the plant works from ~6:00 to ~19:00. ✅ **Zero at night is normal.**

In [ ]:
# Write (or paste) your code here



**What to do — 2) ONE rule for the outliers.** Make the power **missing (NaN)** when it is **above 8.5 MW** (the plant maximum) **or below -0.1 MW** (tiny values like -0.02 at night are normal). Print how many you found. Then **remove the rows where the power is missing** (we never invent the target; also remove rows where `pv_power_lag_24h_mw` is missing).

💡 **Hint:** a condition like `(df["pv_power_mw"] > 8.5) | (df["pv_power_mw"] < -0.1)`; then `df.loc[condition, "pv_power_mw"] = np.nan`; then `df.dropna(subset=[...])` and `reset_index`.

🤖 **Prompt for the AI**
```
In my table df, set pv_power_mw to NaN where it is above 8.5 or below -0.1 (print how many rows). Then drop the rows where pv_power_mw or
pv_power_lag_24h_mw is missing and reset the index. Print the number of rows left. Simple code.
```
⚠️ **Trap:** if the AI suggests "remove all rows with power 0 or negative", **say no**: you would delete every night.

In [ ]:
# Write (or paste) your code here



In [ ]:
# CHECK (given)
print("Rows left:", len(df), "   (expected about 12883)")
print("Power from", df["pv_power_mw"].min(), "to", df["pv_power_mw"].max(), "MW   (expected about -0.04 to 8.5)")
print("Missing values left:", int(df.isna().sum().sum()), "   (expected 0)")

---
# Step 3 — Split in time, train XGBoost, measure the error  *(10 min)*
📘 *Train / validation / test.* The model learns on **training** and is checked on **validation** (data it has never seen). In a time series we **never** split randomly: training = the past, validation = the most recent months.

📘 *Baseline.* A model is useful only if it beats a simple rule. Our baseline: **"the power will be the same as yesterday at this hour"**.

**What to do**
1. **Split in time order:** `train_part` = rows with `timestamp` **before 2024-04-01**; `val_part` = rows **from** 2024-04-01.
2. `feature_cols` = all columns **except** `timestamp` and `pv_power_mw`. Make `X_train`, `y_train`, `X_val`, `y_val`.
3. Train an **`XGBRegressor`** called `model` on the training part: `n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=1`.
4. Predict on the validation part into `pred_val`. Force it between **0 and 8.5** (`np.clip`).
5. Compute **MAE, RMSE and nMAE** (= MAE / 8.5 × 100). Compute the same for the **baseline**: the column `pv_power_lag_24h_mw` of `val_part` ("same hour yesterday").
6. Plot five days (2024-05-20 to 2024-05-24): real, predicted, baseline.

🤖 **Prompt for the AI**
```
Split my table df by time: train_part = timestamp before 2024-04-01, val_part = from 2024-04-01. feature_cols = all columns except timestamp and pv_power_mw.
Create X_train, y_train, X_val, y_val. Train XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=1)
called model. Predict on X_val into pred_val and clip between 0 and 8.5. Print MAE, RMSE and nMAE (MAE/8.5*100) of the model, and the same for the baseline
val_part["pv_power_lag_24h_mw"]. Then plot five days (2024-05-20 to 2024-05-24) of real, predicted and baseline. Simple code, no functions.
```
**Think about it (slide 2):** does XGBoost beat the baseline? By how much? Is an error of ~1 % of the plant size acceptable for the grid operator?

In [ ]:
# Write (or paste) your code here



---
# Step 4 — SHAP: what does the model look at?  *(8 min)*

📘 *Explainable AI.* SHAP tells how much **each feature pushed each prediction up or down**. In the plot: **one dot = one hour**, **colour** = value of the feature (red = high, blue = low), **left/right** = the prediction goes down/up.

🤖 **Prompt for the AI**
```
I have a trained XGBoost model called model and a validation table X_val. With SHAP: create a TreeExplainer, take 800 random rows of X_val
(random_state=SEED) as X_sample, compute shap_values and draw the summary plot. Simple code.
```
**Think about it (slide 3):** which feature is at the **top**? Does it make sense for a solar plant? Which features do almost **nothing** (does the sun care about Sundays)? ⚠️ 📘 *Explaining is not proving causality.*

In [ ]:
# Write (or paste) your code here



---
# Step 5 — Save the model and take the final exam  *(7 min)*

📘 *Deployment.* We retrain on **all** the clean data (so the model knows the latest months), then **save** the model and the **names of the features** in the right order.

**Use exactly these file names:** `pv_model.joblib` and `pv_feature_cols.json`.

🤖 **Prompt for the AI**
```
Retrain an XGBRegressor (same parameters as before, random_state=SEED, n_jobs=1) called model_final on all rows of df: X_all = df[feature_cols], y_all = df["pv_power_mw"].
Save it with joblib to "pv_model.joblib" and save the list feature_cols with json to "pv_feature_cols.json". Simple code.
```

In [ ]:
# Write (or paste) your code here



### 🏁 The final test
**Wait for your instructor.** When the link of the test file is given, paste it in `TEST_URL`. Then run it (no coding here).

In [ ]:
# STEP 5 (part 2) - THE TEST. Run it only when the instructor says so.
TEST_URL = "PASTE THE LINK GIVEN BY THE INSTRUCTOR"

# load what we saved
model_loaded = joblib.load("pv_model.joblib")
model_loaded.set_params(n_jobs=1)
with open("pv_feature_cols.json") as f:
    feature_cols_loaded = json.load(f)

# load the test data and prepare it EXACTLY like the training data
test = pd.read_csv(TEST_URL)
test["timestamp"] = pd.to_datetime(test["timestamp"])
test = test.sort_values("timestamp").reset_index(drop=True)
test["hour"] = test["timestamp"].dt.hour

# predict (same columns, same order as in training)
pred_test = model_loaded.predict(test[feature_cols_loaded])
pred_test = np.clip(pred_test, 0, 8.5)

# errors
y = test["pv_power_mw"].values
mae  = mean_absolute_error(y, pred_test)
rmse = np.sqrt(mean_squared_error(y, pred_test))
nmae = mae / 8.5 * 100

baseline = test["pv_power_lag_24h_mw"].values
mae_b  = mean_absolute_error(y, baseline)
rmse_b = np.sqrt(mean_squared_error(y, baseline))
nmae_b = mae_b / 8.5 * 100

print("TEST                  MAE (MW)   RMSE (MW)   nMAE (%)")
print(f"XGBoost model         {mae:8.3f}   {rmse:9.3f}   {nmae:8.2f}")
print(f"Baseline (yesterday)  {mae_b:8.3f}   {rmse_b:9.3f}   {nmae_b:8.2f}")

week = (test["timestamp"] >= "2024-08-05") & (test["timestamp"] < "2024-08-12")
plt.plot(test.loc[week, "timestamp"], y[week.values], label="Real")
plt.plot(test.loc[week, "timestamp"], pred_test[week.values], label="Predicted")
plt.legend(); plt.ylabel("MW"); plt.title("Test: one week in August 2024"); plt.show()

---
## 🎤 Your 3 slides
Use the AI to help with the format, but **the content must be yours**.

| Slide | Content | Questions |
|---|---|---|
| **1. Data** | the problems you found and what you did | Which problems? Which thing did you **not** touch (night zeros) and why? |
| **2. Results** | validation and test errors vs the baseline | Does the model beat the baseline? Is the test similar to the validation? |
| **3. What the model looks at** | the SHAP plot | What matters most? Does it match physics? What would you monitor in operation? |

---
## ⭐ Bonus (if you finish early): write it yourself with the AI
Ask ChatGPT / Claude for a **SHAP dependence plot** of `ghi_forecast_wm2` coloured by `temperature_c`:
```
I have a trained XGBoost model, shap_values and a table X_sample. Draw a SHAP dependence plot of "ghi_forecast_wm2" with interaction_index "temperature_c". Simple code.
```
Do you see that, with the same sun, **hotter hours produce a bit less**? (Hot panels are less efficient.)